<a href="https://colab.research.google.com/github/saksaw/AI-Model-Evaluation-and-Data-Quality-Suite/blob/main/LLM_Evaluation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

print("🚀 Initializing Programmatic LLM Evaluation Engine (No-API Edition)...")

# ==========================================
# 1. THE DATASET (User Queries, Context, and LLM Responses)
# ==========================================
# This matches the core structure of modern AI quality evaluation datasets.
dataset = [
    {
        "query": "What is the return window for defective electronics?",
        "context": "Our policy states customers can return electronics within 30 days for a full refund. Defective items are eligible for free return shipping.",
        "llm_output": "You can return defective electronics within 30 days, and return shipping is completely free."
    },
    {
        "query": "Does the company ship to India?",
        "context": "We currently offer international shipping exclusively to Canada, the United Kingdom, and Australia.",
        "llm_output": "Yes! We ship worldwide to all international countries including India and Asia."
        # INACCURATE: This is a classic LLM "Hallucination" based on the provided context!
    }
]

# ==========================================
# 2. EVALUATION LOGIC CORE
# ==========================================
def evaluate_faithfulness(output, context):
    """
    Measures if the LLM output is strictly grounded in the provided context.
    (Catches Hallucinations)
    """
    output_words = set(output.lower().replace(".", "").replace(",", "").split())
    context_words = set(context.lower().replace(".", "").replace(",", "").split())

    # Exclude common stop words to focus on factual terms
    stop_words = {'is', 'the', 'and', 'a', 'to', 'for', 'in', 'can', 'with', 'our', 'we', 'you'}
    output_facts = output_words - stop_words
    context_facts = context_words - stop_words

    # Calculate what % of the LLM's factual words actually exist in the source document
    matching_facts = output_facts.intersection(context_facts)
    if not output_facts: return 0.0

    return len(matching_facts) / len(output_facts)

def evaluate_answer_relevance(output, query):
    """
    Measures if the LLM output directly answers the specific intent of the user's query.
    """
    output_words = set(output.lower().split())
    query_words = set(query.lower().split())
    stop_words = {'what', 'is', 'the', 'for', 'does', 'to', 'a', 'do'}

    query_intents = query_words - stop_words
    matching_intents = output_words.intersection(query_intents)

    if not query_intents: return 0.0
    return len(matching_intents) / len(query_intents)

# ==========================================
# 3. RUNNING THE AUTOMATED AUDIT
# ==========================================
evaluation_results = []
FAITHFULNESS_THRESHOLD = 0.75

for case in dataset:
    faith_score = evaluate_faithfulness(case["llm_output"], case["context"])
    relevance_score = evaluate_answer_relevance(case["llm_output"], case["query"])

    # An output passes ONLY if it doesn't hallucinate (high faithfulness score)
    status = "PASS" if faith_score >= FAITHFULNESS_THRESHOLD else "FAIL (Hallucination Detected)"

    evaluation_results.append({
        "User Query": case["query"],
        "Faithfulness Score": round(faith_score, 2),
        "Relevance Score": round(relevance_score, 2),
        "Audit Status": status
    })

# ==========================================
# 4. EXPORTING THE QUALITY REPORT
# ==========================================
df_report = pd.DataFrame(evaluation_results)
print("\n📊 ENTERPRISE AI DATA QUALITY AUDIT REPORT:")
print("=========================================================")
print(df_report.to_string(index=False))

🚀 Initializing Programmatic LLM Evaluation Engine (No-API Edition)...

📊 ENTERPRISE AI DATA QUALITY AUDIT REPORT:
                                          User Query  Faithfulness Score  Relevance Score                  Audit Status
What is the return window for defective electronics?                0.89             0.50                          PASS
                     Does the company ship to India?                0.11             0.33 FAIL (Hallucination Detected)
